<a href="https://colab.research.google.com/github/jolineuichanco/DataAnalytics/blob/main/Week06_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Week 6 Demo**


In [ ]:
import pandas as pd

data_dict = {
    'Month': ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'],
    'Sales': [102, 110, 112, 115, 117, 116, 118, 129, 123, 120, 129, 133],
    'Ad_Dollars': [5.5, 5.8, 6.0, 5.8, 6.2, 6.3, 6.5, 7.0, 6.5, 6.4, 6.7, 6.8]
}

df = pd.DataFrame(data_dict)
df.head()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.relplot(data=df, x='Ad_Dollars', y='Sales', kind='scatter')
plt.xlabel('Ad Dollars')
plt.ylabel('Sales')
plt.title('Ad Dollars vs Sales')
plt.show()

## Sklearn

In [ ]:
from sklearn import linear_model

# create an instance of the LinearRegression class
reg = linear_model.LinearRegression()

# Note the double square brackets for the "X"
# (because you can have multiple columns in X)
reg.fit(df[['Ad_Dollars']], df['Sales'])

# Get the slope and intercept from the fitted model
slope = reg.coef_[0]
intercept = reg.intercept_

print("Fitted line has slope " + str(slope))
print("Fitted line has slope " + str(intercept))

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.regplot(data=df, x='Ad_Dollars', y='Sales', order=1)
plt.xlabel('Ad Dollars')
plt.ylabel('Sales')
plt.title('Ad Dollars vs Sales with Regression Line')

# Create the equation string
equation = f'y = {slope:.2f}x + {intercept:.2f}'

# Add the equation to the plot
plt.text(0.05, 0.95, equation, transform=plt.gca().transAxes, fontsize=12, verticalalignment='top', bbox=dict(boxstyle='round,pad=0.5', fc='yellow', alpha=0.5))

plt.show()

In [ ]:
from sklearn.metrics import r2_score

# Get predictions from the fitted model
y_pred = reg.predict(df[['Ad_Dollars']])

# Compute the R2 score based on actual and predicted y values
r2 = r2_score(df['Sales'], y_pred)
print(f"R-squared score: {r2}")

## Statsmodels

In [ ]:
import statsmodels.api as sm

# Our model needs an intercept so we add a column of 1s
x = sm.add_constant(df['Ad_Dollars']) # Use x = df['Ad_Dollars'] if no constant in model
y = df['Sales']

model = sm.OLS(y, x)
results = model.fit()

print(results.summary())

## Checking Assumptions

In [ ]:
# Load data
url = "https://raw.githubusercontent.com/vincentarelbundock/Rdatasets/master/csv/HistData/Guerry.csv"
dat = pd.read_csv(url)
dat.head()

In [ ]:
import statsmodels.formula.api as smf
import numpy as np

# Fit regression model (using the natural log of one of the regressors)
results = smf.ols("Lottery ~ Literacy + np.log(Pop1831)", data=dat).fit()

# Calculate fitted values for the current model (results from 'dat')
fitted = results.fittedvalues

# residuals
residuals = results.resid

# Inspect the results
print(results.summary())

In [ ]:
import seaborn as sns
plt.figure(figsize=(5, 3.5))
sns.histplot(residuals, bins=15)

In [ ]:
import statsmodels.api as sm
import matplotlib.pyplot as plt

# Create the Q-Q plot with an explicit figure and axes for sizing control
fig, ax = plt.subplots(figsize=(5, 3.5))
sm.qqplot(results.resid, line='s', ax=ax)
plt.title('Q-Q Plot of Residuals')
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(5, 3.5))
# Plot residuals against fitted values to check for heteroskedasticity
# Goal: Residuals should be randomly scattered around zero with no clear pattern (e.g., no 'megaphone' shape)
sns.residplot(x=fitted, y=residuals, lowess=True, line_kws={'color': 'red'})
plt.xlabel('Fitted Values')
plt.ylabel('Residuals')
plt.title('Residuals vs Fitted Values')
plt.show()

In [ ]:
from statsmodels.stats.outliers_influence import OLSInfluence
import matplotlib.pyplot as plt

# Calculate Cook's distance
cook_distance = OLSInfluence(results).cooks_distance[0]

# Plot Cook's distance
fig = plt.figure(figsize=(5, 3.5))
plt.stem(cook_distance, markerfmt=",")
plt.xlabel('Observation Index')
plt.ylabel('Cook\'s Distance')
plt.title('Cook\'s Distance Plot')

# Add a reference line for potential cutoff (e.g., 4 / n or 1)
n = len(dat) # Number of observations
plt.axhline(y=4/n, color='red', linestyle='--', label=f'Threshold (4/n = {4/n:.3f})')
plt.legend()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot residuals vs. Literacy
plt.figure(figsize=(10, 3.5))
plt.subplot(1, 2, 1) # 1 row, 2 columns, first plot
sns.residplot(x=dat['Literacy'], y=residuals, lowess=True, line_kws={'color': 'red'})
plt.xlabel('Literacy')
plt.ylabel('Residuals')
plt.title('Residuals vs. Literacy')

# Plot residuals vs. log(Pop1831)
plt.subplot(1, 2, 2) # 1 row, 2 columns, second plot
sns.residplot(x=dat['Pop1831'].apply(np.log), y=residuals, lowess=True, line_kws={'color': 'red'})
# lowess line shows the local average of teh residuals
plt.xlabel('log(Pop1831)')
plt.ylabel('Residuals')
plt.title('Residuals vs. log(Pop1831)')

plt.tight_layout()
plt.show()

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
import pandas as pd

# Get the exogenous variables (predictors) from the model
# 'model.exog' contains the predictors, including the constant
x_vars = results.model.exog

# Create a DataFrame to store VIF values
vif_data = pd.DataFrame()
vif_data["Variable"] = results.model.exog_names
vif_data["VIF"] = [variance_inflation_factor(x_vars, i) for i in range(x_vars.shape[1])]

display(vif_data)

# **In-Class Prediction Session 1**

In [ ]:
import matplotlib.pyplot as plt

exam_third = [65, 67, 71, 71, 66, 75, 67, 70, 71, 69, 69]
exam_final = [175, 133, 185, 163, 126, 198, 153, 163, 159, 151, 159]

plt.scatter(x=exam_third, y=exam_final)
plt.xlabel('Exam Third Score')
plt.ylabel('Exam Final Score')
plt.title('Exam Third vs. Exam Final Scores')
plt.show()

In [ ]:
import pandas as pd

exam_data = {
    'Exam_Third_Score': exam_third,
    'Exam_Final_Score': exam_final
}

exam_df = pd.DataFrame(exam_data)
exam_df.describe()

In [ ]:
import statsmodels.formula.api as smf
import numpy as np

# Fit regression model (using the natural log of one of the regressors)
results = smf.ols("Exam_Final_Score ~ Exam_Third_Score", data=exam_df).fit()

# Calculate fitted values for the current model (results from 'dat')
fitted = results.fittedvalues

# residuals
residuals = results.resid

# Inspect the results
print(results.summary())

In [ ]:
import seaborn as sns

sns.regplot(data=exam_df, x='Exam_Third_Score', y='Exam_Final_Score')


In [ ]:
# Plot residuals vs. Exam_Third_Score
sns.residplot(x=exam_df['Exam_Third_Score'], y=residuals)
#plt.xlabel('Literacy')
#plt.ylabel('Residuals')
#plt.title('Residuals vs. Literacy')

# **In-Class Prediction Session 2**

In [ ]:
import pandas as pd

url = "https://vincentarelbundock.github.io/Rdatasets/csv/datasets/mtcars.csv"
mtcars = pd.read_csv(url, index_col=0)

# Display the first few rows
mtcars.head()


In [ ]:
import seaborn as sns

sns.regplot(data=mtcars, x='mpg', y='hp', order=1)